# Tema de programare: Investigarea valorilor aberante


Bun venit la tema despre **Investigarea valorilor aberante**!

Valorile aberante - puncte de date care se abat semnificativ de la majoritate - pot distorsiona antrenarea modelelor, pot umfla metricile de eroare si pot ascunde tiparele reale. Sa stii *cand* si *cum* sa le detectezi este o abilitate esentiala de curatare a datelor.

Acest notebook acopera trei strategii complementare de detectare:

| Metoda | Cand sa o folosesti |
|--------|-------------|
| IQR | Distributii asimetrice, robusta la extreme |
| Z-score | Distributii aproximativ normale |
| Isolation Forest | Date cu dimensionalitate mare, fara presupuneri despre distributie |

**Ce vei face in aceasta tema:**

* Vei implementa detectarea valorilor aberante bazata pe **IQR** folosind limite quartilice
* Vei implementa detectarea valorilor aberante bazata pe **Z-score** folosind abaterea standard
* Vei implementa detectarea valorilor aberante cu **Isolation Forest** pentru date multivariate
* Vei elimina valorile aberante detectate si vei returna un DataFrame curat
* Vei crea un **raport rezumativ al valorilor aberante** cuprinzator pentru toate coloanele numerice

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod care este in afara acestor comentarii**.

* Poti adauga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, asa ca nu te baza pe celulele nou create pentru a gazdui codul solutiei tale; foloseste locurile oferite pentru asta.

---


## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea datelor](#1---understanding-the-data)
- [2 - Detectarea valorilor aberante cu IQR](#2---iqr-outlier-detection)
    - **[Exercitiul 1 - `detect_iqr_outliers`](#exercise-1---detect_iqr_outliers)**
- [3 - Detectarea valorilor aberante cu Z-Score](#3---z-score-outlier-detection)
    - **[Exercitiul 2 - `detect_zscore_outliers`](#exercise-2---detect_zscore_outliers)**
- [4 - Detectarea valorilor aberante cu Isolation Forest](#4---isolation-forest-outlier-detection)
    - **[Exercitiul 3 - `detect_isolation_forest_outliers`](#exercise-3---detect_isolation_forest_outliers)**
- [5 - Eliminarea valorilor aberante](#5---removing-outliers)
    - **[Exercitiul 4 - `remove_outliers`](#exercise-4---remove_outliers)**
- [6 - Raport rezumativ al valorilor aberante](#6---outlier-summary-report)
    - **[Exercitiul 5 - `create_outlier_summary`](#exercise-5---create_outlier_summary)**
- [7 - Vizualizarea rezultatelor](#7---visualizing-results)


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Intelegerea datelor

Generam un set de date sintetic de vanzari cu valori aberante injectate, astfel incat sa poti verifica functiile tale de detectare raportandu-le la un ground truth. Setul de date contine coloane numerice care reprezinta metrici de vanzari; o fractiune din randuri a fost deplasata in mod deliberat catre valori extreme.

Ruleaza celulele de mai jos pentru a incarca datele si a inspecta distributia lor inainte de a scrie orice logica de detectare.


In [ ]:
df = helper_utils.generate_sales_dataset(n_samples=300, outlier_rate=0.05, random_state=42)
print(f"Dataset shape: {df.shape}")
print(f"\nBasic statistics:")
df.describe().round(2)

In [ ]:
helper_utils.visualize_outliers_boxplot(df)

<a name='2---iqr-outlier-detection'></a>
## 2 - Detectarea valorilor aberante cu IQR

Metoda **Interquartile Range (IQR)** marcheaza o valoare ca fiind aberanta atunci cand se afla cu mai mult de `factor × IQR` sub Q1 sau peste Q3. Cu valoarea implicita `factor=1.5`, limitele sunt:

$$\text{lower} = Q_1 - 1.5 \times \text{IQR}, \qquad \text{upper} = Q_3 + 1.5 \times \text{IQR}$$

Aceasta metoda este robusta deoarece quartilele in sine nu sunt influentate de valori extreme.


<a name='exercise-1---detect_iqr_outliers'></a>
### **Exercitiul 1 - `detect_iqr_outliers`**

**Sarcina ta:**
Implementeaza o functie care detecteaza valorile aberante intr-o singura coloana a unui DataFrame folosind metoda IQR.

**Cerinte:**
- Returneaza un **`pd.Series` boolean** cu acelasi index ca `df`, unde `True` marcheaza o valoare aberanta.
- Calculeaza Q1 si Q3 folosind metoda `.quantile()` la `0.25` si `0.75`.
- Calculeaza IQR ca `Q3 - Q1`.
- Marcheaza orice valoare sub `Q1 - factor * IQR` sau peste `Q3 + factor * IQR`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Extrage coloana tinta: `series = df[column]`.
2. Calculeaza quartilele:
   ```python
   Q1 = series.quantile(0.25)
   Q3 = series.quantile(0.75)
   ```
3. Calculeaza IQR: `IQR = Q3 - Q1`.
4. Defineste limitele:
   ```python
   lower = Q1 - factor * IQR
   upper = Q3 + factor * IQR
   ```
5. Returneaza o masca booleana: `(series < lower) | (series > upper)`.

</details>


In [ ]:
# GRADED FUNCTION: detect_iqr_outliers
def detect_iqr_outliers(df, column, factor=1.5):
    """
    Detect outliers using the Interquartile Range (IQR) method.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column name to check for outliers.
        factor (float): IQR multiplier for bounds (default 1.5).

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
   ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### ÎNCEPUT DE COD AICI ###

In [ ]:
iqr_mask = detect_iqr_outliers(df, 'sales_amount')
print(f"IQR outliers in sales_amount: {iqr_mask.sum()}")
print(f"Outlier percentage: {iqr_mask.mean()*100:.1f}%")

In [ ]:
unittests.exercise_1(detect_iqr_outliers)

<a name='3---z-score-outlier-detection'></a>
## 3 - Detectarea valorilor aberante cu Z-Score

Metoda **Z-score** masoara cate abateri standard se afla o valoare fata de media coloanei:

$$z = \frac{x - \mu}{\sigma}$$

O valoare este marcata atunci cand $|z| > \text{threshold}$ (de obicei 3.0). Aceasta metoda functioneaza cel mai bine cand coloana are o distributie aproximativ normala.


<a name='exercise-2---detect_zscore_outliers'></a>
### **Exercitiul 2 - `detect_zscore_outliers`**

**Sarcina ta:**
Implementeaza detectarea valorilor aberante bazata pe Z-score-ul absolut al unei coloane.

**Cerinte:**
- Returneaza un **`pd.Series` boolean** aliniat la `df.index`.
- Gestioneaza valorile `NaN`: elimina-le inainte de a calcula Z-score-urile, dar pastreaza randurile lor ca `False` in output.
- Foloseste `scipy.stats.zscore` pentru calculul scorului.
- Marcheaza valorile al caror Z-score absolut depaseste `threshold`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Elimina NaN-urile: `col_data = df[column].dropna()`.
2. Calculeaza Z-score-urile: `z_scores = np.abs(stats.zscore(col_data))`.
3. Initializeaza o masca `False` peste intregul index al DataFrame-ului:
   ```python
   mask = pd.Series(False, index=df.index)
   ```
4. Asigneaza rezultatele doar pozitiilor non-NaN:
   ```python
   mask[col_data.index] = z_scores > threshold
   ```
5. Returneaza `mask`.

</details>


In [ ]:
# GRADED FUNCTION: detect_zscore_outliers
def detect_zscore_outliers(df, column, threshold=3.0):
    """
    Detect outliers using Z-score method.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to analyze.
        threshold (float): Z-score threshold (default 3.0).

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
   ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### ÎNCEPUT DE COD AICI ###

In [ ]:
zscore_mask = detect_zscore_outliers(df, 'sales_amount')
print(f"Z-score outliers in sales_amount: {zscore_mask.sum()}")

In [ ]:
unittests.exercise_2(detect_zscore_outliers)

<a name='4---isolation-forest-outlier-detection'></a>
## 4 - Detectarea valorilor aberante cu Isolation Forest

**Isolation Forest** este un algoritm de detectare a anomaliilor bazat pe arbori, care functioneaza prin partitionarea aleatoare a spatiului caracteristicilor. Anomaliile sunt izolate mai repede (sunt necesare mai putine split-uri) decat punctele normale, asa ca primesc un scor de anomalie mai mic.

Spre deosebire de IQR sau Z-score, Isolation Forest opereaza in **spatiu multivariat**: ia in considerare toate coloanele selectate simultan, ceea ce il face ideal pentru detectarea valorilor aberante care sunt neobisnuite in combinatie, chiar daca fiecare valoare individuala pare normala.


<a name='exercise-3---detect_isolation_forest_outliers'></a>
### **Exercitiul 3 - `detect_isolation_forest_outliers`**

**Sarcina ta:**
Implementeaza detectarea multivariata a valorilor aberante folosind `IsolationForest` din `sklearn`.

**Cerinte:**
- Daca `columns` este `None`, foloseste implicit toate coloanele numerice.
- Elimina randurile cu valori NaN in coloanele selectate inainte de antrenare.
- Returneaza un **`pd.Series` boolean** aliniat la `df.index`; randurile excluse din cauza valorilor NaN trebuie sa fie `False`.
- `IsolationForest.fit_predict` returneaza `-1` pentru valori aberante si `1` pentru valori normale.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Selecteaza coloanele numerice daca `columns is None`:
   ```python
   columns = df.select_dtypes(include='number').columns.tolist()
   ```
2. Extrage si elimina randurile cu NaN: `X = df[columns].dropna()`.
3. Antreneaza si prezice:
   ```python
   iso = IsolationForest(contamination=contamination, random_state=random_state)
   preds = iso.fit_predict(X)
   ```
4. Construieste un Series `False` peste `df.index`, apoi seteaza `preds == -1` pentru randurile din `X.index`.

</details>


In [ ]:
# GRADED FUNCTION: detect_isolation_forest_outliers
def detect_isolation_forest_outliers(df, columns=None, contamination=0.1, random_state=42):
    """
    Detect outliers using Isolation Forest algorithm.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to use (default: all numeric).
        contamination (float): Expected proportion of outliers.
        random_state (int): Random seed.

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
   ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### ÎNCEPUT DE COD AICI ###

In [ ]:
iso_mask = detect_isolation_forest_outliers(df)
print(f"Isolation Forest outliers: {iso_mask.sum()}")

In [ ]:
unittests.exercise_3(detect_isolation_forest_outliers)

<a name='5---removing-outliers'></a>
## 5 - Eliminarea valorilor aberante

Odata ce ai identificat ce randuri sunt valori aberante, trebuie sa le elimini curat si sa returnezi un DataFrame pregatit pentru modelare ulterioara. Cerintele-cheie sunt:
- Pastreaza doar randurile unde masca este `False`.
- Reseteaza indexul integer astfel incat DataFrame-ul curatat sa inceapa de la 0.


<a name='exercise-4---remove_outliers'></a>
### **Exercitiul 4 - `remove_outliers`**

**Sarcina ta:**
Implementeaza o functie care elimina randurile cu valori aberante dintr-un DataFrame folosind o masca booleana.

**Cerinte:**
- Pastreaza randurile unde `outlier_mask` este `False`.
- Reseteaza indexul DataFrame-ului returnat (`drop=True`).
- **Nu** modifica DataFrame-ul original.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Inverseaza masca cu `~outlier_mask` pentru a selecta randurile care nu sunt valori aberante.
2. Filtreaza: `df[~outlier_mask]`.
3. Reseteaza indexul: `.reset_index(drop=True)`.
4. Returneaza rezultatul intr-o singura expresie.

</details>


In [ ]:
# GRADED FUNCTION: remove_outliers
def remove_outliers(df, outlier_mask):
    """
    Remove rows identified as outliers.

    Args:
        df (pd.DataFrame): Input DataFrame.
        outlier_mask (pd.Series): Boolean mask where True means outlier.

    Returns:
        pd.DataFrame: DataFrame with outlier rows removed and index reset.
    """
   ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### ÎNCEPUT DE COD AICI ###

In [ ]:
iqr_mask_demo = detect_iqr_outliers(df, 'sales_amount')
df_no_outliers = remove_outliers(df, iqr_mask_demo)
print(f"Original shape : {df.shape}")
print(f"Cleaned shape  : {df_no_outliers.shape}")
print(f"Rows removed   : {df.shape[0] - df_no_outliers.shape[0]}")

In [ ]:
unittests.exercise_4(remove_outliers)

<a name='6---outlier-summary-report'></a>
## 6 - Raport rezumativ al valorilor aberante

Un raport rezumativ le ofera stakeholderilor o vedere rapida si usor de interpretat asupra calitatii datelor. Aici vei construi o functie care ruleaza detectarea IQR pe fiecare coloana numerica, uneste marcajele de valori aberante si raporteaza statistici agregate.


<a name='exercise-5---create_outlier_summary'></a>
### **Exercitiul 5 - `create_outlier_summary`**

**Sarcina ta:**
Creeaza un dictionar rezumativ al statisticilor privind valorile aberante in toate coloanele numerice folosind metoda IQR.

**Cerinte:**
- Daca `columns` este `None`, foloseste toate coloanele numerice.
- Un rand este numarat ca valoare aberanta daca este marcat in **orice** coloana (uniunea mastilor pe coloane).
- Returneaza un `dict` cu exact aceste chei:
  - `'total_outliers'` - numarul de randuri cu valori aberante (int)
  - `'clean_rows'` - numarul de randuri fara valori aberante (int)
  - `'outlier_pct'` - procentul de randuri cu valori aberante rotunjit la 2 zecimale (float)

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Selecteaza coloanele numerice daca este nevoie.
2. Initializeaza un Series `False`: `all_outliers = pd.Series(False, index=df.index)`.
3. Pentru fiecare coloana, calculeaza masca IQR si combina folosind `|`:
   ```python
   all_outliers = all_outliers | mask
   ```
4. Numara: `total = int(all_outliers.sum())`.
5. Construieste si returneaza dict-ul:
   ```python
   return {
       'total_outliers': total,
       'clean_rows': len(df) - total,
       'outlier_pct': round(total / len(df) * 100, 2),
   }
   ```

</details>


In [ ]:
# GRADED FUNCTION: create_outlier_summary
def create_outlier_summary(df, columns=None):
    """
    Create a summary report of outliers across all numeric columns.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to check (default: all numeric).

    Returns:
        dict: {'total_outliers': int, 'clean_rows': int, 'outlier_pct': float}
    """
   ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### ÎNCEPUT DE COD AICI ###

In [ ]:
summary = create_outlier_summary(df)
print("Outlier Summary Report")
print("=======================")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
unittests.exercise_5(create_outlier_summary)

<a name='7---visualizing-results'></a>
## 7 - Vizualizarea rezultatelor

Sa folosim functiile pe care le-ai implementat pentru a curata setul de date si pentru a compara vizual distributia inainte si dupa eliminarea valorilor aberante.


In [ ]:
iqr_mask_final = detect_iqr_outliers(df, 'sales_amount')
df_cleaned = remove_outliers(df, iqr_mask_final)
helper_utils.plot_outlier_comparison(df, df_cleaned, 'sales_amount')

## Felicitari!

Ai finalizat cu succes tema despre **Investigarea valorilor aberante**. Iata un rezumat al ceea ce ai implementat:

| Exercitiu | Functie | Metoda |
|----------|----------|--------|
| 1 | `detect_iqr_outliers` | Regula gardurilor IQR |
| 2 | `detect_zscore_outliers` | Z-score absolut |
| 3 | `detect_isolation_forest_outliers` | Isolation Forest |
| 4 | `remove_outliers` | Filtrare cu masca booleana |
| 5 | `create_outlier_summary` | Raport agregat |

Ai acum un set robust de instrumente pentru detectarea valorilor aberante, care acopera atat metode statistice univariate, cat si abordari multivariate bazate pe machine learning. Aceste abilitati se vor transfera direct catre pipeline-uri reale de curatare a datelor!
